# Regression Model Bake-off

Goal: compare global regressors for cross-sectional pairs-trading residuals.
**No regime classifier here — purely a regression-quality study.**

**Models**:
1. **ElasticNet** (baseline, also used for feature selection)
2. **Natural cubic splines** (sklearn `SplineTransformer` + Ridge)
3. **GAM** (`pyGAM` `LinearGAM` with smooth terms)
4. **Hybrid**: Ridge for global trend + spline (NS) on residuals
5. **GP** (Gaussian Process): `DotProduct` (global linear trend, robust extrapolation
   when peers leave their historical range) **+** `RBF` (local non-linear curvature)
   **+** `WhiteKernel` (noise). Captures non-linear peer→target relationships while
   degrading gracefully to the linear trend out-of-range.

**Two training regimes** (both with exp-decay weights, α=0.995):
- **Rolling**: fit on the last 200 bars only.
- **Expanding**: fit on all history up to t (0→t).
  (GP additionally caps training to the most-recent `GP_MAX_TRAIN` rows — sklearn's
   GPR has no sample-weight support, so this stands in for the exp-decay recency and
   bounds the O(n³) cost.)

**Feature selection**: ElasticNet on each refit batch → keep top-K predictors by |coef|.
Every model — GP included — receives exactly these selected features.

**Metrics**:
- Residual stationarity: ADF p-value
- 1-day predictive R²
- **H=20 forecast quality**: MAE / RMSE / R² for predicting the residual 20 days ahead
  (we feed today's selected features and ask the model what the residual will be in 20 days
   by predicting price 20 days ahead vs realized price)

Targets: the 10 fixed sector anchors. Refit every 50 bars.

In [ ]:
# Cell 2 — Imports + cache
import os, sys, math, pickle, warnings, time
from pathlib import Path
from datetime import datetime
import random as _random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.preprocessing import StandardScaler, SplineTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, DotProduct, WhiteKernel, ConstantKernel
HAS_GP = True

try:
    from pygam import LinearGAM, s
    HAS_PYGAM = True
except Exception as e:
    HAS_PYGAM = False; print('pyGAM not available:', e)

try:
    from statsmodels.tsa.stattools import adfuller
except Exception:
    adfuller = None

warnings.filterwarnings('ignore')
%matplotlib inline

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'strategy').exists():
    cand = Path(r'C:\algo-trading-project')
    if cand.exists(): PROJECT_ROOT = cand
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

SEED = 42
_random.seed(SEED); np.random.seed(SEED)

FORCE_RECOMPUTE = False
CACHE_DIR = Path('outputs/regression_bakeoff'); CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PKL = CACHE_DIR / 'all_results.pkl'
print(f'pyGAM: {HAS_PYGAM} | GP: {HAS_GP} | cache: {CACHE_DIR}')

In [ ]:
# Cell 3 — Project setup, data, splits, sector pools
from config import SECTORS
from strategy.strategy_config import StrategyConfig
from strategy.pipeline import StrategyPipeline
from strategy.splits import chrono_split

cfg = StrategyConfig(force_recompute=False, make_plots=False)
pipeline = StrategyPipeline(cfg)
md = pipeline.load_data()
split = chrono_split(md.prices.index, cfg)
train_idx = pd.DatetimeIndex(split.train_idx).sort_values()
val_idx   = pd.DatetimeIndex(split.val_idx).sort_values()
test_idx  = pd.DatetimeIndex(split.test_idx).sort_values()

TARGETS = []  # (sector_name, target_ticker, peer_list)
for etf, s in SECTORS.items():
    tgt = s['target']
    peers = [p for p in s['predictors'] if p in md.prices.columns]
    if tgt not in md.prices.columns or not peers: continue
    TARGETS.append((s['name'], tgt, peers))

for name, tgt, peers in TARGETS:
    print(f'  {name:18s} target={tgt:6s} peers={len(peers)}')

In [ ]:
# Cell 4 — Constants + helpers
RETRAIN_EVERY  = 50
MIN_TRAIN_DAYS = 100
ROLL_WIN       = 200       # rolling-window size for the rolling regime
DECAY_ALPHA    = 0.995     # exp-decay sample weights
TOP_K_FEATS    = 5         # number of features kept by feature selection
FORECAST_H     = 20        # days ahead for predictive metric
RESID_Z_WIN    = 60
SPLINE_K       = 4         # knots per feature for NS
SPLINE_DEG     = 3
RIDGE_ALPHA    = 1.0
EN_ALPHA       = 0.01
EN_L1_RATIO    = 0.5
GP_MAX_TRAIN   = 400       # cap GP train rows (GPR is O(n^3)); keep most-recent (recency)
GP_NOISE       = 0.1       # base observation-noise variance for the weighted GP (see fit_gp)

def decay_weights(n, alpha=DECAY_ALPHA):
    w = alpha ** np.arange(n-1, -1, -1, dtype=float); return w / w.sum() * n

def safe_adf(s, fallback=1.0):
    s = pd.Series(s).dropna()
    if len(s) < 30 or adfuller is None: return fallback
    try:    return float(adfuller(s, autolag='AIC')[1])
    except Exception: return fallback

def feature_select(X, y, w, top_k=TOP_K_FEATS):
    """ElasticNet feature selection: return indices of top |coef| features."""
    sc = StandardScaler().fit(X)
    en = ElasticNet(alpha=EN_ALPHA, l1_ratio=EN_L1_RATIO, max_iter=5000, random_state=SEED)
    en.fit(sc.transform(X), y, sample_weight=w)
    coef = np.abs(en.coef_)
    if not np.any(coef > 0):
        return np.arange(min(top_k, X.shape[1]))
    return np.argsort(-coef)[:top_k]

# --- Model factories: each returns a callable that takes (X_train, y_train, w) and returns predictor f(X) ---
def fit_elasticnet(X, y, w):
    sc = StandardScaler().fit(X)
    m  = ElasticNet(alpha=EN_ALPHA, l1_ratio=EN_L1_RATIO, max_iter=5000, random_state=SEED)
    m.fit(sc.transform(X), y, sample_weight=w)
    return lambda Xq: m.predict(sc.transform(Xq))

def fit_ns(X, y, w):
    sc = StandardScaler().fit(X)
    Xs = sc.transform(X)
    sp = SplineTransformer(n_knots=SPLINE_K, degree=SPLINE_DEG, knots='quantile',
                           extrapolation='linear').fit(Xs)
    Z = sp.transform(Xs)
    m = Ridge(alpha=RIDGE_ALPHA, random_state=SEED).fit(Z, y, sample_weight=w)
    return lambda Xq: m.predict(sp.transform(sc.transform(Xq)))

def fit_gam(X, y, w):
    if not HAS_PYGAM: raise RuntimeError('pyGAM unavailable')
    sc = StandardScaler().fit(X)
    Xs = sc.transform(X)
    terms = s(0)
    for i in range(1, X.shape[1]): terms = terms + s(i)
    m = LinearGAM(terms, max_iter=200)
    m.fit(Xs, y, weights=w)
    return lambda Xq: m.predict(sc.transform(Xq))

def fit_hybrid(X, y, w):
    """Ridge global trend + NS spline on Ridge residuals."""
    sc = StandardScaler().fit(X)
    Xs = sc.transform(X)
    ridge = Ridge(alpha=RIDGE_ALPHA, random_state=SEED).fit(Xs, y, sample_weight=w)
    trend = ridge.predict(Xs); resid = y - trend
    sp = SplineTransformer(n_knots=SPLINE_K, degree=SPLINE_DEG, knots='quantile',
                           extrapolation='linear').fit(Xs)
    Z = sp.transform(Xs)
    spline = Ridge(alpha=RIDGE_ALPHA*5.0, random_state=SEED).fit(Z, resid, sample_weight=w)
    def predict(Xq):
        Xqs = sc.transform(Xq)
        return ridge.predict(Xqs) + spline.predict(sp.transform(Xqs))
    return predict

def fit_gp(X, y, w):
    """Gaussian Process with exp-decay recency weighting — the same DECAY_ALPHA
    logic the linear/spline models apply via sample_weight.

    sklearn's GaussianProcessRegressor has no sample_weight, so we encode the
    weights as per-sample observation noise: alpha_i = GP_NOISE / w_i. A recent
    day (large w) -> small noise -> trusted; an older day (small w) -> large
    noise -> discounted. This is the GP/GLS equivalent of weighted least squares.
    WhiteKernel is dropped because `alpha` now carries the (heteroscedastic)
    noise. GP_MAX_TRAIN caps the O(n^3) cost by keeping the most-recent rows
    (which already hold the largest weights)."""
    if not HAS_GP: raise RuntimeError('GP unavailable')
    w = np.asarray(w, dtype=float)
    if len(y) > GP_MAX_TRAIN:
        X = X[-GP_MAX_TRAIN:]; y = y[-GP_MAX_TRAIN:]; w = w[-GP_MAX_TRAIN:]
    w = w / w.mean()                                   # mean 1 -> alpha stays well-scaled
    alpha = GP_NOISE / np.clip(w, 1e-6, None)          # recency -> heteroscedastic noise
    sc = StandardScaler().fit(X)
    Xs = sc.transform(X)
    kernel = (ConstantKernel(1.0, (1e-2, 1e3)) * DotProduct(sigma_0=1.0)
              + ConstantKernel(1.0, (1e-2, 1e3)) * RBF(length_scale=1.0, length_scale_bounds=(1e-1, 1e2)))
    gp = GaussianProcessRegressor(kernel=kernel, alpha=alpha, normalize_y=True,
                                  n_restarts_optimizer=0, random_state=SEED)
    gp.fit(Xs, y)
    return lambda Xq: gp.predict(sc.transform(Xq))

MODEL_FACTORIES = {
    'ElasticNet': fit_elasticnet,
    'NS_spline':  fit_ns,
    'GAM':        fit_gam if HAS_PYGAM else None,
    'Hybrid':     fit_hybrid,
    'GP':         fit_gp if HAS_GP else None,
}
MODEL_FACTORIES = {k: v for k, v in MODEL_FACTORIES.items() if v is not None}
print('Models enabled:', list(MODEL_FACTORIES))
print(f'Refit every {RETRAIN_EVERY} bars | rolling_win={ROLL_WIN} | top_k={TOP_K_FEATS} | H={FORECAST_H} | gp_max_train={GP_MAX_TRAIN} gp_noise={GP_NOISE}')

In [ ]:
# Cell 5 — Walk-forward driver: refit every RETRAIN_EVERY, both rolling+expanding
def walk_forward(X_df, y_ser, model_fit, regime='expanding'):
    """Returns DataFrame with cols: pred_1, pred_h, residual, selected_feats.
    For every test index t, predict y[t] and y[t+H] using model fit on history.
    Feature selection (ElasticNet top-K) done at each refit.
    """
    idx = X_df.index
    n = len(idx)
    pred1 = np.full(n, np.nan)
    pred_h = np.full(n, np.nan)
    feat_sel_log = [None] * n
    last_refit = -10**9
    predict_fn = None
    sel_idx = None
    cols = list(X_df.columns)
    Xv = X_df.values; yv = y_ser.values
    for t in range(n):
        need_refit = (t >= MIN_TRAIN_DAYS) and (t - last_refit >= RETRAIN_EVERY)
        if need_refit:
            if regime == 'expanding':
                lo = 0
            else:
                lo = max(0, t - ROLL_WIN)
            slc = slice(lo, t)
            Xt = Xv[slc]; yt = yv[slc]
            mask = (~np.isnan(Xt).any(axis=1)) & (~np.isnan(yt))
            if mask.sum() < MIN_TRAIN_DAYS:
                continue
            Xt = Xt[mask]; yt = yt[mask]
            w = decay_weights(len(yt))
            try:
                sel_idx = feature_select(Xt, yt, w)
                predict_fn = model_fit(Xt[:, sel_idx], yt, w)
                last_refit = t
            except Exception as e:
                predict_fn = None
                continue
        if predict_fn is not None and sel_idx is not None:
            if not np.isnan(Xv[t]).any():
                try:
                    pred1[t]  = float(predict_fn(Xv[t:t+1, sel_idx])[0])
                except Exception:
                    pass
            # h-step ahead: predict using TODAY's features but compare to y[t+H]
            if t + FORECAST_H < n and not np.isnan(Xv[t]).any():
                try:
                    pred_h[t] = float(predict_fn(Xv[t:t+1, sel_idx])[0])
                except Exception:
                    pass
            feat_sel_log[t] = [cols[i] for i in sel_idx]
    return pd.DataFrame({
        'pred_1': pred1,
        'pred_h': pred_h,
        'features': feat_sel_log,
    }, index=idx)

print('walk_forward ready.')

In [ ]:
# Cell 6 — Run all (model × regime × target). Incremental cache: only compute
# combos missing from the pickle, so adding a new model (e.g. GP) does NOT
# recompute the existing ones.
# Set RECOMPUTE_MODELS = {'GP'} to force-refresh just one model after changing it.
RECOMPUTE_MODELS = set()

ALL = {}
if RESULTS_PKL.exists() and not FORCE_RECOMPUTE:
    with open(RESULTS_PKL, 'rb') as f: ALL = pickle.load(f)
    print(f'[CACHE] loaded {len(ALL)} entries')

t0 = time.time(); n_new = 0
for sec_name, tgt, peers in TARGETS:
    X_df = md.prices[peers].copy(); y_ser = md.prices[tgt].copy()
    common = X_df.dropna().index.intersection(y_ser.dropna().index)
    X_df = X_df.loc[common]; y_ser = y_ser.loc[common]
    for mname, mfit in MODEL_FACTORIES.items():
        for regime in ('rolling', 'expanding'):
            key = (sec_name, tgt, mname, regime)
            if key in ALL and not FORCE_RECOMPUTE and mname not in RECOMPUTE_MODELS:
                continue
            tk = time.time()
            res = walk_forward(X_df, y_ser, mfit, regime=regime)
            # build residual + h-residual
            price = y_ser.reindex(res.index)
            price_h = price.shift(-FORECAST_H)
            residual   = price - res['pred_1']
            residual_h = price_h - res['pred_h']
            resz = (residual - residual.rolling(RESID_Z_WIN).mean()) / residual.rolling(RESID_Z_WIN).std()
            ALL[key] = pd.DataFrame({
                'price':       price,
                'pred_1':      res['pred_1'],
                'pred_h':      res['pred_h'],
                'price_h':     price_h,
                'residual':    residual,
                'residual_h':  residual_h,
                'residual_z':  resz,
            })
            n_new += 1
            print(f'  {sec_name:18s} {tgt:6s} {mname:10s} {regime:9s} {time.time()-tk:5.1f}s')

if n_new:
    with open(RESULTS_PKL, 'wb') as f: pickle.dump(ALL, f)
    print(f'Saved -> {RESULTS_PKL.name}. New entries: {n_new}. Total: {time.time()-t0:.1f}s')
else:
    print(f'No new entries to compute. {len(ALL)} cached.')

In [ ]:
# Cell 7 — Metrics table (per target × model × regime, on VAL window)
def metric_row(df, window_idx, label):
    sub = df.loc[df.index.isin(window_idx)].dropna(subset=['price','pred_1'])
    if sub.empty: return None
    r2_1 = r2_score(sub['price'], sub['pred_1'])
    mae_1 = mean_absolute_error(sub['price'], sub['pred_1'])
    rmse_1 = math.sqrt(mean_squared_error(sub['price'], sub['pred_1']))
    adfp = safe_adf(sub['residual'])
    rz_std = float(sub['residual_z'].std())
    sub_h = df.loc[df.index.isin(window_idx)].dropna(subset=['price_h','pred_h'])
    r2_h = mae_h = rmse_h = np.nan
    if not sub_h.empty:
        r2_h = r2_score(sub_h['price_h'], sub_h['pred_h'])
        mae_h = mean_absolute_error(sub_h['price_h'], sub_h['pred_h'])
        rmse_h = math.sqrt(mean_squared_error(sub_h['price_h'], sub_h['pred_h']))
    return dict(window=label, n=len(sub), r2_1=r2_1, mae_1=mae_1, rmse_1=rmse_1,
                adf_p=adfp, rz_std=rz_std, r2_h=r2_h, mae_h=mae_h, rmse_h=rmse_h)

rows = []
for (sec, tgt, mname, regime), df in ALL.items():
    for label, widx in [('TRAIN', train_idx), ('VAL', val_idx), ('TEST', test_idx)]:
        r = metric_row(df, widx, label)
        if r is None: continue
        rows.append({'sector': sec, 'target': tgt, 'model': mname, 'regime': regime, **r})
metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(CACHE_DIR / 'metrics.csv', index=False)
print(f'Metrics: {len(metrics_df)} rows -> metrics.csv')

# Aggregate: mean across targets, focused on VAL
agg = (metrics_df[metrics_df['window']=='VAL']
       .groupby(['model','regime'])[['r2_1','mae_1','rmse_1','adf_p','rz_std','r2_h','mae_h','rmse_h']]
       .mean().round(4))
print('\n=== VAL aggregate (mean across 10 targets) ===')
print(agg.to_string())

agg_test = (metrics_df[metrics_df['window']=='TEST']
            .groupby(['model','regime'])[['r2_1','mae_1','rmse_1','adf_p','rz_std','r2_h','mae_h','rmse_h']]
            .mean().round(4))
print('\n=== TEST aggregate (mean across 10 targets) ===')
print(agg_test.to_string())

In [ ]:
# Cell 8 — Plot 1: residual time series for one representative target (each model × regime)
DEMO_TARGET = ('Financials', 'JPM')  # change to inspect a different target
demo_keys = [k for k in ALL.keys() if (k[0], k[1]) == DEMO_TARGET]
# models present in the cache for this target AND known to MODEL_FACTORIES,
# preserving MODEL_FACTORIES order (guards against a stale cache holding a model
# that isn't enabled in this kernel, e.g. GAM when pyGAM is missing).
plot_models = [m for m in MODEL_FACTORIES if any(k[2] == m for k in demo_keys)]
fig, axes = plt.subplots(len(plot_models), 1, figsize=(12, 2.8*len(plot_models)), sharex=True)
if len(plot_models) == 1: axes = [axes]
model_to_ax = {m: axes[i] for i, m in enumerate(plot_models)}
for k in demo_keys:
    sec, tgt, mname, regime = k
    ax = model_to_ax.get(mname)
    if ax is None:        # cached model not enabled in this kernel — skip
        continue
    df = ALL[k].dropna(subset=['residual_z'])
    ax.plot(df.index, df['residual_z'].values, label=regime, linewidth=0.7, alpha=0.85)
    ax.set_title(f'{tgt} — {mname} | residual_z')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axhline( 2.0, color='red', linewidth=0.4, linestyle='--')
    ax.axhline(-2.0, color='red', linewidth=0.4, linestyle='--')
    for split_label, span in [('TRAIN', train_idx), ('VAL', val_idx), ('TEST', test_idx)]:
        ax.axvspan(span[0], span[-1], alpha=0.06,
                   color={'TRAIN':'gray','VAL':'orange','TEST':'green'}[split_label])
    ax.legend(loc='upper left', fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Cell 9 — Plot 2: residual_z histogram per model (TRAIN+VAL combined, expanding regime)
fig, axes = plt.subplots(1, len(MODEL_FACTORIES), figsize=(4.2*len(MODEL_FACTORIES), 3.5), sharey=True)
if len(MODEL_FACTORIES) == 1: axes = [axes]
wnd = train_idx.union(val_idx)
for ax, mname in zip(axes, MODEL_FACTORIES):
    rzs = []
    for k, df in ALL.items():
        if k[2] != mname or k[3] != 'expanding': continue
        rzs.append(df.loc[df.index.isin(wnd), 'residual_z'].dropna().values)
    arr = np.concatenate(rzs) if rzs else np.array([])
    ax.hist(arr, bins=60, density=True, alpha=0.7)
    ax.axvline(0, color='black', linewidth=0.6)
    ax.set_title(f'{mname} (expanding)\nμ={arr.mean():+.2f} σ={arr.std():.2f} |rz|>2={np.mean(np.abs(arr)>2):.2%}')
    ax.set_xlim(-5, 5); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Cell 10 — Plot 3: H=20 forecast scatter (predicted vs realized price), VAL window, expanding
fig, axes = plt.subplots(1, len(MODEL_FACTORIES), figsize=(4.2*len(MODEL_FACTORIES), 4), sharex=False, sharey=False)
if len(MODEL_FACTORIES) == 1: axes = [axes]
for ax, mname in zip(axes, MODEL_FACTORIES):
    xs, ys = [], []
    for k, df in ALL.items():
        if k[2] != mname or k[3] != 'expanding': continue
        sub = df.loc[df.index.isin(val_idx)].dropna(subset=['pred_h','price_h'])
        if sub.empty: continue
        # Normalize per-target so we can stack across targets
        scale = sub['price_h'].std() or 1.0
        center = sub['price_h'].mean()
        xs.append((sub['pred_h'].values - center) / scale)
        ys.append((sub['price_h'].values - center) / scale)
    x = np.concatenate(xs) if xs else np.array([])
    y = np.concatenate(ys) if ys else np.array([])
    if len(x):
        ax.scatter(x, y, s=2, alpha=0.25)
        lim = float(np.nanpercentile(np.abs(np.concatenate([x,y])), 99))
        ax.plot([-lim, lim], [-lim, lim], color='red', linewidth=0.7)
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
        r2 = r2_score(y, x)
        ax.set_title(f'{mname} (H=20, VAL)\nR²={r2:.3f}')
    ax.set_xlabel('pred (z)'); ax.set_ylabel('actual (z)')
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Cell 11 — Plot 4: rolling vs expanding head-to-head bar chart (VAL R²_h per target)
models = list(MODEL_FACTORIES)
tgts   = [t for (_, t, _) in TARGETS]
fig, axes = plt.subplots(1, len(models), figsize=(4*len(models), 4), sharey=True)
if len(models) == 1: axes = [axes]
for ax, mname in zip(axes, models):
    sub = metrics_df[(metrics_df['model']==mname) & (metrics_df['window']=='VAL')]
    if sub.empty: continue
    pv = sub.pivot_table(index='target', columns='regime', values='r2_h')
    pv.reindex(tgts).plot(kind='bar', ax=ax, width=0.8)
    ax.set_title(f'{mname} — VAL R²(H={FORECAST_H})')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_ylabel('R²' if ax is axes[0] else '')
    ax.tick_params(axis='x', rotation=60)
    ax.grid(alpha=0.3, axis='y'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Cell 12 — Plot 5: actual price vs each model's pred_1 for DEMO_TARGET, VAL window (expanding)
fig, ax = plt.subplots(figsize=(13, 5))
demo_keys_exp = [k for k in ALL.keys() if (k[0], k[1])==DEMO_TARGET and k[3]=='expanding']
if demo_keys_exp:
    any_df = ALL[demo_keys_exp[0]]
    val_df = any_df.loc[any_df.index.isin(val_idx)]
    ax.plot(val_df.index, val_df['price'].values, color='black', linewidth=1.2, label='actual')
for k in demo_keys_exp:
    _, _, mname, _ = k
    sub = ALL[k].loc[ALL[k].index.isin(val_idx)]
    ax.plot(sub.index, sub['pred_1'].values, linewidth=0.9, alpha=0.85, label=f'{mname}')
ax.set_title(f'{DEMO_TARGET[1]} — actual vs 1-step predictions (VAL, expanding)')
ax.set_ylabel('price'); ax.legend(loc='best', fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Cell 13 — Best-of summary (single winner per metric on VAL)
val = metrics_df[metrics_df['window']=='VAL'].copy()
agg_target = val.groupby(['model','regime'])
summary = pd.DataFrame({
    'mean_r2_1' : agg_target['r2_1'].mean(),
    'mean_r2_h' : agg_target['r2_h'].mean(),
    'mean_adf_p': agg_target['adf_p'].mean(),
    'mean_rz_std': agg_target['rz_std'].mean(),
    'frac_adf_stationary': agg_target['adf_p'].apply(lambda s: (s < 0.05).mean()),
}).round(4).sort_values('mean_r2_h', ascending=False)
print('=== VAL summary (sorted by mean R²(H=20)) ===')
print(summary.to_string())
summary.to_csv(CACHE_DIR / 'summary_val.csv')

best_pred = summary['mean_r2_h'].idxmax()
best_resid = summary['frac_adf_stationary'].idxmax()
print(f'\nBest by H=20 prediction R²    : {best_pred}')
print(f'Best by residual stationarity : {best_resid}')

In [ ]:
# Cell 14 — Dense forecast-horizon grid (R^2 for h=1..15) using cached pred_1
HORIZONS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 15]

def horizon_metrics(df, window_idx, h):
    sub = df.loc[df.index.isin(window_idx)].dropna(subset=['pred_1','price']).copy()
    sub['actual_h'] = sub['price'].shift(-h)
    sub = sub.dropna(subset=['actual_h'])
    if len(sub) < 20: return None
    y_true, y_pred = sub['actual_h'].values, sub['pred_1'].values
    ss_res = float(((y_true - y_pred) ** 2).sum())
    ss_tot = float(((y_true - y_true.mean()) ** 2).sum())
    r2  = 1.0 - ss_res / ss_tot if ss_tot > 0 else float('nan')
    mae = float(abs(y_true - y_pred).mean())
    rmse = float(((y_true - y_pred) ** 2).mean() ** 0.5)
    return dict(h=h, n=len(sub), r2=r2, mae=mae, rmse=rmse)

rows = []
for (sec, tgt, mname, regime), df in ALL.items():
    for label, widx in [('TRAIN', train_idx), ('VAL', val_idx), ('TEST', test_idx)]:
        for h in HORIZONS:
            r = horizon_metrics(df, widx, h)
            if r is None: continue
            rows.append({'sector': sec, 'target': tgt, 'model': mname,
                         'regime': regime, 'window': label, **r})
horizon_df = pd.DataFrame(rows)
horizon_df.to_csv(CACHE_DIR / 'horizon_metrics.csv', index=False)
print(f'Wrote horizon_metrics.csv ({len(horizon_df)} rows)')

for win in ('VAL', 'TEST'):
    pv = (horizon_df[horizon_df['window']==win]
          .groupby(['model','regime','h'])['r2'].mean()
          .unstack('h').round(3))
    pv = pv[HORIZONS]
    print(f'\n=== {win} — mean R^2 across 10 targets, per horizon ===')
    print(pv.to_string())


In [ ]:
# Cell 15 — Heatmap: model x horizon, mean R^2 on VAL (+ TEST)
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, win in zip(axes, ('VAL', 'TEST')):
    pv = (horizon_df[horizon_df['window']==win]
          .groupby(['model','regime','h'])['r2'].mean()
          .unstack('h'))
    pv = pv[HORIZONS]
    pv.index = [f'{m}|{r}' for (m, r) in pv.index]
    pv = pv.sort_values(pv.columns[0], ascending=False)
    arr = pv.values.astype(float)
    vmax = float(np.nanmax(np.abs(arr))) if np.isfinite(arr).any() else 1.0
    im = ax.imshow(arr, aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_xticks(range(len(HORIZONS))); ax.set_xticklabels(HORIZONS)
    ax.set_yticks(range(len(pv.index))); ax.set_yticklabels(pv.index, fontsize=8)
    ax.set_title(f'{win}  mean R^2(h)  across 10 targets')
    ax.set_xlabel('horizon (days)')
    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            v = arr[i, j]
            if np.isfinite(v):
                ax.text(j, i, f'{v:+.2f}', ha='center', va='center',
                        fontsize=7, color='black' if abs(v) < vmax*0.5 else 'white')
    fig.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout(); plt.show()


In [ ]:
# Cell 16 — Line plot: R^2 vs horizon, one panel per model
fig, axes = plt.subplots(1, len(MODEL_FACTORIES), figsize=(4.2*len(MODEL_FACTORIES), 4), sharey=True)
if len(MODEL_FACTORIES) == 1: axes = [axes]
for ax, mname in zip(axes, MODEL_FACTORIES):
    for regime in ('rolling', 'expanding'):
        for win, style in [('VAL', '-'), ('TEST', '--')]:
            sub = horizon_df[(horizon_df['model']==mname) &
                             (horizon_df['regime']==regime) &
                             (horizon_df['window']==win)]
            if sub.empty: continue
            ser = sub.groupby('h')['r2'].mean().reindex(HORIZONS)
            ax.plot(ser.index, ser.values, style, marker='o', markersize=3,
                    label=f'{regime}-{win}', linewidth=1.2, alpha=0.85)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_title(f'{mname}'); ax.set_xlabel('horizon (days)')
    if ax is axes[0]: ax.set_ylabel('mean R^2 across targets')
    ax.grid(alpha=0.3); ax.legend(fontsize=7, loc='best')
plt.tight_layout(); plt.show()


In [ ]:
# Cell 17 — GP vs others: focused leaderboard + GP rank per metric
# Reads metrics_df (Cell 7). Higher r2 is better; lower rz_std / adf_p is better.
def leaderboard(window):
    w = metrics_df[metrics_df['window'] == window]
    return (w.groupby(['model', 'regime'])
              .agg(r2_1=('r2_1', 'mean'),
                   r2_h=('r2_h', 'mean'),
                   rz_std=('rz_std', 'mean'),
                   adf_p=('adf_p', 'mean'),
                   frac_stationary=('adf_p', lambda s: (s < 0.05).mean()))
              .round(4))

for win in ('VAL', 'TEST'):
    lb = leaderboard(win)
    print(f'\n================ {win}: leaderboard (sorted best-first by r2_1) ================')
    print(lb.sort_values('r2_1', ascending=False).to_string())

# Where does GP land? rank 1 = best across all model x regime combos (TEST)
test_lb = leaderboard('TEST')
n = len(test_lb)
print(f'\n================ GP rank on TEST  (1 = best of {n} model x regime combos) ================')
# metric -> ascending? (True means smaller is better)
for metric, ascending in [('r2_1', False), ('r2_h', False), ('rz_std', True), ('adf_p', True)]:
    ranks = test_lb[metric].rank(ascending=ascending, method='min')
    for (model, regime), rk in ranks.items():
        if model == 'GP':
            print(f'  {metric:8s}  GP-{regime:9s}  value={test_lb.loc[(model, regime), metric]:+.4f}  rank={int(rk)}/{n}')


## Analysis & Verdict — GP vs the others

*(Snapshot from the run on 2026-06-12, **GP now uses exp-decay recency weighting** —
`alpha_i = GP_NOISE / w_i`, the GP equivalent of the `sample_weight` the other models use.
Re-run the cells above to refresh the numbers.)*

### Two caveats before the numbers
1. **GAM is not in this comparison** — `pygam` is not installed in this kernel, so `MODEL_FACTORIES`
   dropped it. The bake-off here is **ElasticNet / NS_spline / Hybrid / GP**. Install `pygam` and
   re-run Cells 4→6 to add GAM back.
2. **No model produces stationary residuals out-of-sample** — `frac_adf_stationary = 0.0` and
   `adf_p ≈ 1.0` for *every* model on VAL and TEST. This is a property of the set-up
   (cross-sectional price-**level** regression extrapolates poorly), not something specific to GP.

### What the decay weighting did to GP (TEST)
| metric | GP·rolling before → after | GP·expanding before → after |
|--------|:-------------------------:|:---------------------------:|
| R²(1-day) | +0.103 → **+0.173** | −0.030 → **+0.046** |
| R²(H=20)  | −0.574 → **−0.515** | −0.705 → **−0.578** |
| rz_std    | 1.247 → 1.246       | 1.253 → **1.226** |

The recency weighting improved GP on **every** metric — exactly the intended effect (older days count less).

### Out-of-sample (TEST) — current leaderboard

| model · regime          | R²(1-day) | R²(H=20) | rz_std |
|-------------------------|:---------:|:--------:|:------:|
| **NS_spline · rolling** | **+0.219** 🥇 | **−0.367** 🥇 | **1.226** 🥇 |
| **GP · rolling**        | **+0.173** 🥈 | −0.515 | 1.246 |
| Hybrid · rolling        | +0.064 | −0.497 🥈 | 1.258 |
| GP · expanding          | +0.046 | −0.578 | 1.226 |
| NS_spline · expanding   | +0.024 | −0.564 | 1.236 |
| ElasticNet · rolling    | +0.019 | −0.606 | 1.271 |

On VAL by R²(1-day): NS_spline·rolling (+0.078) → **GP·rolling (+0.065)** → ElasticNet → **GP·expanding** → …

### Verdict
- **NS_spline (rolling) still wins every metric** — best 1-day R², H=20 R², rz_std, and it stays
  R²-positive out to h≈9 on TEST. Cheapest, too.
- **GP·rolling is now a strong, clear 2nd.** The decay weighting was the missing piece: TEST 1-day R²
  jumped +0.103 → **+0.173**, shrinking the gap to the spline from 0.116 to **0.046**. GP·rolling now
  holds positive R² out to h=5 (was h=3), and GP·expanding ties the best `rz_std` (1.226).
- **GP confirms a real non-linear peer→target signal** (it clearly beats the linear ElasticNet and now
  Hybrid on 1-day R²), but it is far more expensive (~30s/target expanding vs near-instant).

**Practical takeaway:** with recency weighting GP is genuinely competitive — the spline still wins on
both accuracy and cost, but the margin is now small enough that tuning `GP_NOISE` / `length_scale`
could plausibly push GP past it. For production today, NS_spline·rolling remains the pick; GP is a
serious challenger worth one tuning pass.